# Balancin: Real-Time PID Tuning Dashboard

Welcome to the interactive tuning dashboard for the Balancin project. 

This environment allows you to tune the Proportional, Integral, and Derivative ($K_p$, $K_i$, $K_d$) constants of the system in real-time. The control logic runs in a dedicated background thread, meaning you can adjust parameters and monitor the system's step response without needing to stop the motor or restart the software.

### 1. System Initialization
Run the cell below to load the FPGA bitstream, map the custom hardware drivers, and initialize the background control manager.<

In [1]:
import sys
import os

# Add the correct absolute path of the project to the Python search path
sys.path.append('/home/xilinx/jupyter_notebooks/PabloSanchez/pwm/Overlay')

from balancin.overlay import BalancinOverlay
from balancin.balancer import Balancer

# Load the FPGA bitstream (the BalancinOverlay class automatically finds the bitstream file now)
ol = BalancinOverlay()

# Initialize the threaded PID manager (defaulting to safe starting values)
bal = Balancer(ol, kp=3.5, ki=0.0, kd=0.0)

print("Hardware loaded and Balancer initialized successfully.")
print("Waiting for control panel launch...")

Hardware loaded and Balancer initialized successfully.
Waiting for control panel launch...


### 2. Control Panel Configuration

Below we configure the graphical interface for the tuning parameters. 
- **Sliders:** Allow real-time adjustment of the PID constants and the target angle.
- **Controls:** Safely start and stop the hardware control thread.

*(Note: This cell initializes the layout in memory. The actual interface will be rendered in the final dashboard view).*

In [2]:
import ipywidgets as widgets

# --- 1. Style and Layout Definitions ---
# This ensures all sliders line up perfectly and look professional
widget_style = {'description_width': '100px'}
widget_layout = widgets.Layout(width='400px')

# --- 2. Create the Sliders ---
kp_slider = widgets.FloatSlider(value=3.5, min=0.0, max=20.0, step=0.1, 
                                description='Kp (Prop):', style=widget_style, layout=widget_layout)
ki_slider = widgets.FloatSlider(value=0.0, min=0.0, max=10.0, step=0.01, 
                                description='Ki (Integ):', style=widget_style, layout=widget_layout)
kd_slider = widgets.FloatSlider(value=0.0, min=0.0, max=5.0, step=0.01, 
                                description='Kd (Deriv):', style=widget_style, layout=widget_layout)
target_slider = widgets.FloatSlider(value=20.0, min=-45.0, max=45.0, step=1.0, 
                                    description='Target Angle:', style=widget_style, layout=widget_layout)

# --- 3. Create the Buttons ---
btn_start = widgets.Button(description='Start Motor', button_style='success', icon='play', layout=widgets.Layout(width='195px'))
btn_stop = widgets.Button(description='Stop Motor', button_style='danger', icon='stop', layout=widgets.Layout(width='195px'))

# --- 4. Define Callbacks ---
def on_slider_change(change):
    # Push the new values directly into the running thread
    bal.set_params(
        kp=kp_slider.value,
        ki=ki_slider.value,
        kd=kd_slider.value,
        target=target_slider.value
    )

def on_start_clicked(b):
    bal.start()

def on_stop_clicked(b):
    bal.stop()

# --- 5. Attach Callbacks ---
kp_slider.observe(on_slider_change, names='value')
ki_slider.observe(on_slider_change, names='value')
kd_slider.observe(on_slider_change, names='value')
target_slider.observe(on_slider_change, names='value')

btn_start.on_click(on_start_clicked)
btn_stop.on_click(on_stop_clicked)

# --- 6. Group into a Layout ---
button_box = widgets.HBox([btn_start, btn_stop])
control_panel = widgets.VBox([
    widgets.HTML("<b>PID Tuning Parameters</b>"),
    kp_slider, ki_slider, kd_slider, 
    widgets.HTML("<hr><b>Reference Control</b>"),
    target_slider,
    widgets.HTML("<br>"),
    button_box
])

print("Control panel layout configured.")

Control panel layout configured.


### 3. Telemetry Plot Configuration

To monitor the step response and stability of the PID controller, we configure a live telemetry plot. 
This plot uses dynamic data updating (rather than full canvas redraws) to provide a smooth, flicker-free visualization of the hardware's behavior over time.

In [3]:
import matplotlib.pyplot as plt

# --- 1. Create the Figure and Axes ---
fig, ax = plt.subplots(figsize=(8, 4))

# --- 2. Initialize Empty Lines ---
# By saving these line objects to variables, we can inject new data into them later
line_angle, = ax.plot([], [], label='Current Angle', color='blue', lw=2)
line_target, = ax.plot([], [], label='Target', linestyle='--', color='red', lw=2)

# --- 3. Configure Plot Aesthetics ---
ax.set_ylim(-45, 45)       # Fixed Y-axis for stability
ax.set_xlim(0, 10)         # Initial X-axis window
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (°)')
ax.set_title('Live System Telemetry')
ax.legend(loc='upper right')
ax.grid(True, linestyle=':', alpha=0.7)

# --- 4. Prevent Immediate Rendering ---
# We close the static plot here so we can display it dynamically in our custom layout
plt.close(fig)

print("Telemetry plot layout configured.")

Telemetry plot layout configured.


### 4. Launch Unified Dashboard

The block below launches the complete tuning interface. 

**Instructions:**
1. Click **Start Motor** to engage the hardware loop.
2. Use the sliders to tune the PID response in real-time.
3. The graph will update automatically to show the system's step response.
4. To safely exit the tuning mode, click **Stop Motor** and interrupt the Jupyter cell (the Stop button ◼️ in the toolbar).

In [4]:
import asyncio
from IPython.display import display, clear_output

# --- 1. Create a dedicated output area for the plot ---
plot_output = widgets.Output()

# --- 2. Assemble the Layout ---
# Place the control panel on the left, and the plot on the right
dashboard = widgets.HBox([control_panel, plot_output])

# Render the interface
display(dashboard)

# --- 3. The Non-Blocking Live UI Update Loop ---
# Using asyncio allows Jupyter's event loop to process widget clicks and sliders
async def update_plot():
    try:
        while True:
            # Check if the background thread has collected data
            if len(bal.history) >= 2:
                # Extract data from the deque safely
                data = list(bal.history)
                times = [d[0] - data[0][0] for d in data] # Normalize start time to 0
                angles = [d[1] for d in data]
                targets = [d[2] for d in data]
                
                # Update the lines with new data (highly efficient)
                line_angle.set_data(times, angles)
                line_target.set_data(times, targets)
                
                # Scroll the X-axis to keep the last 10 seconds in view
                current_time = times[-1]
                ax.set_xlim(max(0, current_time - 10), max(10, current_time))
                
                # Push the updated figure to the output container
                with plot_output:
                    clear_output(wait=True)
                    display(fig)
                    
            # Yield control back to Jupyter so widgets stay responsive (10 FPS)
            await asyncio.sleep(0.1)
            
    except asyncio.CancelledError:
        print("Dashboard display loop halted.")

# Stop any previous plot task if the cell is run multiple times
try:
    plot_task.cancel()
except NameError:
    pass

# Start the async loop without blocking the cell
plot_task = asyncio.create_task(update_plot())

Balancer thread started.
Balancer stopped and motor safety shutdown complete.
Balancer stopped and motor safety shutdown complete.
Balancer thread started.
